## 📦 安装依赖库

这个步骤用于安装运行 MusicGen 所需的核心库，包括：

- `librosa`：音频处理
- `encodec`：音频编码器（Meta 提供）
- `transformers`：加载 MusicGen 模型
- `accelerate`：加速推理
- `soundfile`：音频读写
- `gradio`：后续可用于做 Web UI（可选）

使用 `-q` 参数可以减少安装日志输出，让界面更清爽。

In [1]:
!pip install -q librosa encodec transformers accelerate soundfile gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 48.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## ⚙️ 检查运行环境（GPU）

这个步骤用于确认当前运行环境是否启用了 GPU。

MusicGen 模型较大，如果使用 CPU 生成会非常慢，因此推荐使用 GPU（如 Colab 的 T4 / A100）。

如果显示 GPU 不可用，请在：
`Runtime → Change runtime type → Hardware accelerator → GPU`
进行设置。

In [2]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU is NOT available. Please ensure your Colab runtime type is set to GPU.")

GPU is available!
GPU name: Tesla T4


## 🤖 加载 MusicGen 模型

这里我们加载：

- `AutoProcessor`：用于处理文本输入
- `MusicgenForConditionalGeneration`：MusicGen 音乐生成模型

使用的是：`facebook/musicgen-medium`

说明：
- `medium` 是中等大小模型（效果与速度平衡）
- 模型会自动加载到 GPU（如果可用）

In [3]:
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import soundfile as sf
import gradio as gr
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")

# 加载处理器和模型
processor = AutoProcessor.from_pretrained("facebook/musicgen-medium")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-medium").to(device)

Using device: cuda


preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/8.04G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.04G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/996 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_encoder.shared.weight to text_encoder.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-medium
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

## 🎼 根据文本生成音乐

在这个步骤中，我们通过“文本提示词（prompt）”生成音乐。

流程如下：

1. 编写音乐描述（例如风格、节奏、乐器）
2. 使用 processor 将文本转换为模型输入
3. 使用 model.generate() 生成音频
4. 转换为 numpy 格式
5. 保存为 `.wav` 文件

关键参数：
- `max_new_tokens`：控制生成时长（越大 → 音乐越长）

示例 prompt：
> Light jazz-inspired piano, optimistic and airy...

你可以修改 prompt 来生成不同风格的音乐 🎵

In [10]:
import torch
import soundfile as sf

# 输入文本描述
prompt = ["bright piano piece | hopeful uplifting mood | light staccato touches | clean articulation"]

# 编码输入
inputs = processor(
    text=prompt,
    padding=True,
    return_tensors="pt"
).to(device)

# 生成音频
with torch.no_grad():
    audio_values = model.generate(
        **inputs,
        max_new_tokens=512  # 控制长度（越大越长）
    )

# 转 numpy
audio = audio_values[0].cpu().numpy()

# 保存为 wav
sf.write("musicgen_medium_output.wav", audio.T, samplerate=32000)

print("生成完成！")

生成完成！


## 🔊 播放生成的音乐

使用 `IPython.display.Audio` 在 Notebook 中直接播放生成的音频。

参数说明：
- `audio`：生成的音频数据
- `rate=32000`：采样率（需与生成时一致）

运行后即可在页面中点击播放 🎧

In [11]:
from IPython.display import Audio

Audio(audio, rate=32000)